In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os

model = YOLO("best.pt")

IMAGE_FOLDER = "akshay_separate.v1i.yolov8/train/images"
data = []

In [ ]:
data = []

for img in os.listdir(IMAGE_FOLDER):

    path = os.path.join(IMAGE_FOLDER, img)

    results = model(path)
    r = results[0]

    if r.masks is None:
        continue

    image = cv2.imread(path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    for mask_tensor in r.masks.data:

        mask = mask_tensor.cpu().numpy()

        mask = cv2.resize(mask, (gray.shape[1], gray.shape[0]))
        mask = mask > 0.5

        pixels = gray[mask]

        if len(pixels) == 0:
            continue

        mean_T = np.mean(pixels)
        std_T = np.std(pixels)

        threshold = mean_T + 1.5 * std_T
        hotspot_ratio = np.sum(pixels > threshold) / len(pixels)

        gradient = cv2.Laplacian(gray, cv2.CV_64F)
        gradient_values = np.abs(gradient[mask])

        gradient_mean = np.mean(gradient_values)

        data.append([
            img,
            mean_T,
            std_T,
            hotspot_ratio,
            gradient_mean
        ])

In [ ]:
df = pd.DataFrame(data, columns=[
    "image",
    "mean_T",
    "std_T",
    "hotspot_ratio",
    "gradient_mean"
])

df.to_csv("defect_feature.csv", index=False)

In [ ]:
df["defect_score"] = (
    0.5 * df["hotspot_ratio"] +
    0.3 * df["gradient_mean"] +
    0.2 * df["std_T"]
)

threshold = df["defect_score"].median()

df["defect"] = df["defect_score"].apply(
        
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[["mean_T","std_T","hotspot_ratio","gradient_mean"]]
y = df["defect"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report
import joblib

pred = model.predict(X_test)

print(classification_report(y_test, pred))

In [ ]:
print("Accuracy:", model.score(X_test, y_test))
joblib.dump(model,"defect_model.pkl")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import joblib

# load models
seg_model = YOLO("best.pt")
clf = joblib.load("defect_model.pkl")

# load image
image_path = "akshay_separate.v1i.yolov8/test/images/FLIR0238_jpg.rf.9a264086df52e2a78494e39a3861493b.jpg"
image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

results = seg_model(image)
r = results[0]

if r.masks is not None:

    for mask_tensor in r.masks.data:

        mask = mask_tensor.cpu().numpy()
        mask = cv2.resize(mask,(gray.shape[1],gray.shape[0]))
        mask = mask > 0.5

        pixels = gray[mask]

        if len(pixels)==0:
            continue

        # compute features
        mean_T = np.mean(pixels)
        std_T = np.std(pixels)

        threshold = mean_T + 1.5*std_T
        hotspot_ratio = np.sum(pixels>threshold)/len(pixels)

        gradient = cv2.Laplacian(gray, cv2.CV_64F)
        gradient_mean = np.mean(np.abs(gradient[mask]))

        features = pd.DataFrame([[mean_T,std_T,hotspot_ratio,gradient_mean]],columns=["mean_T","std_T","hotspot_ratio","gradient_mean"])

        pred = clf.predict(features)[0]

        if pred == 1:
            print("DEFECT")
        else:
            print("NO DEFECT")